# Environment: episodios y pasos

Dos inputs → la misma unidad por paso → dos outputs e historia de episodio.

API **2.1.3**, kernel dedicado y ejecución offline. Los actores Python son operadores deterministas.

In [1]:
from pathlib import Path
import sys
tutorial_root = next(p / "tutorials" for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "tutorials/_offline.py").is_file())
sys.path.insert(0, str(tutorial_root))
from _offline import enforce_offline, blocked_attempts
enforce_offline()
import agentic_systems as toolkit
assert toolkit.__version__ == "2.1.3", "Instala el wheel local 2.1.3: no hay fallback a 2.1.2"


In [2]:
@toolkit.tool
def increment(value: int) -> dict:
    """Suma uno de forma determinista."""
    return {"value": value + 1}

@toolkit.tool
def double(value: int) -> dict:
    """Duplica el valor."""
    return {"value": value * 2}

arithmetic = toolkit.skill(name="arithmetic", tools=[increment, double])
increase = toolkit.Action("increase", "increment")
multiply = toolkit.Action("multiply", "double")

In [3]:
agent = toolkit.Agent("calculator", toolkit.Sequence((increase, multiply)), skills=(arithmetic,))
application = toolkit.System("team", toolkit.Sequence((agent,)))
episode = toolkit.Environment(name="two_steps", unit=application, records=({"value": 3}, {"value": -3}))
result = episode.run()
assert result.ok and result.data["outputs"] == [{"value": 8}, {"value": -4}]
assert result.data["summary"]["memory"] == {"processed": 2}
print({"input": [3, -3], "process": "un paso por registro", "output": result.data["outputs"]})

{'input': [3, -3], 'process': 'un paso por registro', 'output': [{'value': 8}, {'value': -4}]}


Cada run abre un episodio nuevo. Los registros son inputs independientes; la memoria del episodio no se inyecta automáticamente en los Agents. Las transiciones están en `result.data['history']` y las ejecuciones de las unidades en `result.children`.

In [4]:
assert blocked_attempts() == 0
print("PASS · inferencia: 0")

PASS · inferencia: 0
